## RAPTOR

RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval) is an advanced RAG technique that **organizes documents into a tree of summaries**.

It creates summaries of small chunks, then combines those summaries into higher-level summaries.

```text
Documents
    ↓
Chunks
    ↓
Chunk Summaries
    ↓
Higher-Level Summaries
    ↓
RAPTOR Tree
    ↓
Retrieval
```


In [1]:
import uuid
import numpy as np

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sklearn.cluster import KMeans

C:\Users\Acer\AppData\Local\Temp\ipykernel_14532\4128724341.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Euta webpage load gareko
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")

docs = loader.load()

print(f"Total documents: {len(docs)}")

Total documents: 1


In [3]:
# Document lai sano sano chunks ma divide gareko
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = text_splitter.split_documents(docs)

print(f"Total chunks: {len(docs)}")

Total chunks: 36


In [4]:
# Testing ko lagi suruko 5 chunks matra use gareko
docs = docs[:5]

print(f"Using {len(docs)} chunks")

Using 5 chunks


In [5]:
# Local Ollama ko Llama 3 model load gareko
llm = ChatOllama(model="llama3:latest", temperature=0)

In [6]:
# Chunk ko summary generate garne prompt
summary_prompt = ChatPromptTemplate.from_template("""
    Summarize the following document chunk.
    Keep the important information and concepts.

    Document:
    {doc}
    """)

# Summary generate garne chain
summary_chain = (
    {"doc": lambda x: x.page_content} | summary_prompt | llm | StrOutputParser()
)

In [7]:
# Sabai chunks ko summary generate gareko
summaries = summary_chain.batch(docs, config={"max_concurrency": 1})

print(f"Total summaries: {len(summaries)}")

Total summaries: 5


In [8]:
print(summaries[0])

Here's a summary of the document chunk:

**LLM Powered Autonomous Agents**

The concept is to build autonomous agents using Large Language Models (LLMs) as their core controller. This can go beyond generating text and programs, but rather be a powerful general problem solver.

**Agent System Overview**

The LLM-powered agent system consists of three key components:

1. **Planning**: Breaks down large tasks into smaller subgoals for efficient handling of complex tasks.
2. **Memory**: Utilizes both short-term memory (in-context learning) and long-term memory (retaining and recalling information over extended periods).
3. **Tool Use**: Enables the agent to use tools and case studies, such as scientific discovery agents and generative agents simulation.

**Key Concepts**

* Task decomposition
* Self-reflection and refinement
* Short-term and long-term memory
* Tool use and case studies


In [9]:
# Local HuggingFace embedding model load gareko
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [10]:
# Sabai summaries ko embedding generate gareko
summary_embeddings = embeddings.embed_documents(summaries)

print("Total embeddings:", len(summary_embeddings))
print("Embedding dimension:", len(summary_embeddings[0]))

Total embeddings: 5
Embedding dimension: 384


In [11]:
# Embeddings lai numpy array ma convert gareko
embedding_matrix = np.array(summary_embeddings)

# Kati ota cluster banaune bhanera define gareko
n_clusters = min(3, len(summaries))

# Similar summaries lai aauti cluster ma group gareko
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")

cluster_labels = kmeans.fit_predict(embedding_matrix)

print("Cluster labels:")
print(cluster_labels)

Cluster labels:
[2 0 1 1 1]


In [12]:
# Cluster haru lai dictionary ma store gareko
clusters = {}

for i, label in enumerate(cluster_labels):
    clusters.setdefault(label, []).append(summaries[i])

print("Number of clusters:", len(clusters))

for cluster_id, cluster_docs in clusters.items():
    print(f"Cluster {cluster_id}: " f"{len(cluster_docs)} summaries")

Number of clusters: 3
Cluster 2: 1 summaries
Cluster 0: 1 summaries
Cluster 1: 3 summaries


In [13]:
# Related summaries lai combine garera
# higher-level summary banaune prompt
cluster_summary_prompt = ChatPromptTemplate.from_template("""
    Combine the following related summaries into
    one higher-level summary.

    Preserve the important concepts and relationships.

    Summaries:
    {summaries}
    """)

# Higher-level summary generate garne chain
cluster_summary_chain = cluster_summary_prompt | llm | StrOutputParser()

In [14]:
cluster_summaries = []

# Pratek cluster ko summary generate gareko
for cluster_id, cluster_docs in clusters.items():

    # Cluster bhitra ko summaries combine gareko
    combined_text = "\n\n".join(cluster_docs)

    # Higher-level summary generate gareko
    summary = cluster_summary_chain.invoke({"summaries": combined_text})

    cluster_summaries.append(summary)

print(f"Generated {len(cluster_summaries)} " "higher-level summaries")

Generated 3 higher-level summaries


In [15]:
# Higher-level summaries ko pani embedding generate gareko
cluster_embeddings = embeddings.embed_documents(cluster_summaries)

print("Higher-level embeddings:", len(cluster_embeddings))

Higher-level embeddings: 3


In [16]:
def raptor_retrieve(query, k=3):

    # User query ko embedding generate gareko
    query_embedding = embeddings.embed_query(query)

    # Original summary sanga similarity calculate gareko
    leaf_scores = np.dot(summary_embeddings, query_embedding)

    # Higher-level summary sanga similarity calculate gareko
    cluster_scores = np.dot(cluster_embeddings, query_embedding)

    # Sabai bhanda relevant leaf summaries select gareko
    leaf_indices = np.argsort(leaf_scores)[-k:][::-1]

    # Sabai bhanda relevant higher-level summaries select gareko
    cluster_indices = np.argsort(cluster_scores)[-k:][::-1]

    return {
        "leaf": [summaries[i] for i in leaf_indices],
        "clusters": [cluster_summaries[i] for i in cluster_indices],
    }

In [ ]:
# User ko query
query = "What is memory in LLM agents?"

# RAPTOR bata relevant information retrieve gareko
results = raptor_retrieve(query)

print("=== LEAF SUMMARIES ===")

for result in results["leaf"]:
    print("\n", result[:500])

print("\n=== HIGH LEVEL SUMMARIES ===")

for result in results["clusters"]:
    print("\n", result[:500])

=== LEAF SUMMARIES ===

 Here's a summary of the document chunk:

**LLM Powered Autonomous Agents**

The concept is to build autonomous agents using Large Language Models (LLMs) as their core controller. This can go beyond generating text and programs, but rather be a powerful general problem solver.

**Agent System Overview**

The LLM-powered agent system consists of three key components:

1. **Planning**: Breaks down large tasks into smaller subgoals for efficient handling of complex tasks.
2. **Memory**: Utilizes bot

 Here's a summary of the document chunk:

**Tool Use**

A key concept in building an autonomous agent system powered by Large Language Models (LLMs) is using external APIs to supplement missing information. This includes:

* Accessing current information that may not be included in pre-trained model weights
* Executing code to perform specific tasks or calculations
* Utilizing proprietary information sources

This approach enables the agent to leverage additional knowl

In [18]:
# Retrieved context bata final answer generate garne prompt
answer_prompt = ChatPromptTemplate.from_template("""
    Answer the question using the following RAPTOR context.

    Context:
    {context}

    Question:
    {question}

    Answer clearly and accurately.
    """)

# Final answer generate garne chain
answer_chain = answer_prompt | llm | StrOutputParser()

In [19]:
# Leaf ra higher-level summaries lai combine gareko
context = "\n\n".join(results["leaf"] + results["clusters"])

In [20]:
# RAPTOR ko retrieved context use garera answer generate gareko
answer = answer_chain.invoke({"context": context, "question": query})

print(answer)

According to the provided RAPTOR context, Memory is one of the three key components of an LLM-powered autonomous agent system. In this context, Memory utilizes both short-term memory (in-context learning) and long-term memory (retaining and recalling information over extended periods). This suggests that the Memory component plays a crucial role in enabling the agent to learn from its experiences, retain knowledge gained through training, and recall relevant information when needed.
